# Perceived Tabooness of English Swear Words
Analyzes survey data collected in a university linguistics course.  
Participants rated the perceived tabooness of swear words used in three sentence types
(**Invective**, **Emotive**, **Informative**) on a **1–7 scale**.

**Swear words covered:** Fuck · Bitch · Hell · Shit  
**Sample size:** n = 50 respondents


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.gridspec import GridSpec

%matplotlib inline


## 0. Configuration
Map each raw CSV column to its word, sentence type, and a short display label.

In [ ]:
CSV_PATH = "Tabooness_Survey_Responses.csv"

# Map raw column names → structured metadata
SENTENCE_META = {
    # FUCK
    "Go fuck yourself.": {
        "word": "Fuck", "type": "Invective",
        "label": "Go fuck yourself.",
    },
    "I can't wait to fucking graduate.": {
        "word": "Fuck", "type": "Emotive",
        "label": "I can't wait to fucking graduate.",
    },
    "I left the apartment as soon as I could after seeing my roommate fucking someone.": {
        "word": "Fuck", "type": "Informative",
        "label": "…my roommate fucking someone.",
    },
    "The word Fuck itself": {
        "word": "Fuck", "type": "Word itself",
        "label": "The word 'Fuck'",
    },
    # BITCH
    "You are such a bitch.": {
        "word": "Bitch", "type": "Invective",
        "label": "You are such a bitch.",
    },
    "That was a bitch of a test.": {
        "word": "Bitch", "type": "Emotive",
        "label": "That was a bitch of a test.",
    },
    "The dog we thought that was a male turned out to be a bitch.": {
        "word": "Bitch", "type": "Informative",
        "label": "…turned out to be a bitch.",
    },
    "The word Bitch": {
        "word": "Bitch", "type": "Word itself",
        "label": "The word 'Bitch'",
    },
    # HELL
    "Go to hell!": {
        "word": "Hell", "type": "Invective",
        "label": "Go to hell!",
    },
    "That was one hell of a ride!": {
        "word": "Hell", "type": "Emotive",
        "label": "That was one hell of a ride!",
    },
    "In the bible, those who have sinned go to hell after their death. ": {
        "word": "Hell", "type": "Informative",
        "label": "…go to hell after their death.",
    },
    "The word Hell": {
        "word": "Hell", "type": "Word itself",
        "label": "The word 'Hell'",
    },
    # SHIT
    "You piece of shit!": {
        "word": "Shit", "type": "Invective",
        "label": "You piece of shit!",
    },
    "My shitty car stopped working yesterday.": {
        "word": "Shit", "type": "Emotive",
        "label": "My shitty car stopped working yesterday.",
    },
    "I spent a whole hour trying to pick up my dog's shit.": {
        "word": "Shit", "type": "Informative",
        "label": "…pick up my dog's shit.",
    },
}

WORDS = ["Fuck", "Bitch", "Hell", "Shit"]
TYPES = ["Invective", "Emotive", "Informative", "Word itself"]

TYPE_COLORS = {
    "Invective":   "#C0392B",
    "Emotive":     "#E67E22",
    "Informative": "#2980B9",
    "Word itself": "#8E44AD",
}

WORD_COLORS = {
    "Fuck":  "#C0392B",
    "Bitch": "#E67E22",
    "Hell":  "#2980B9",
    "Shit":  "#27AE60",
}


## 1. Load & Reshape Data
Filter out aggregated rows (Average, Maximum, Minimum, etc.) and reshape into long format.

In [ ]:
def load_data(path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (respondents_df, long_df).

    Rows whose Timestamp is not a real date string (e.g. 'Average',
    'Word averages', 'Maximum', 'Minimum') are excluded so they do not
    inflate the sample size or skew the means.
    """
    raw = pd.read_csv(path)

    NON_RESPONDENT_LABELS = {"Average", "Word averages", "Maximum", "Minimum"}
    respondents = raw[
        raw["Timestamp"].notna()
        & ~raw["Timestamp"].isin(NON_RESPONDENT_LABELS)
    ].copy()

    records = []
    for col, meta in SENTENCE_META.items():
        if col in respondents.columns:
            numeric = pd.to_numeric(respondents[col], errors="coerce").dropna()
            for val in numeric:
                records.append({
                    "word":   meta["word"],
                    "type":   meta["type"],
                    "label":  meta["label"],
                    "rating": float(val),
                })

    long = pd.DataFrame(records)
    return respondents, long


raw, long = load_data(CSV_PATH)
print(f"Respondents: {len(raw)}")
print(f"Total rating observations: {len(long)}")
long.head()


## 2. Compute Means
Calculate mean ± SEM for each word × sentence type combination.

In [ ]:
def compute_means(long: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (by_word_type, by_type) aggregated DataFrames."""
    by_word_type = (
        long.groupby(["word", "type"])["rating"]
        .agg(mean="mean", sem="sem", n="count")
        .reset_index()
    )
    by_type = (
        long.groupby("type")["rating"]
        .agg(mean="mean", sem="sem", n="count")
        .reset_index()
    )
    return by_word_type, by_type


by_word_type, by_type = compute_means(long)


## 3. Summary Table
Mean tabooness ratings (1 = not taboo at all, 7 = extremely taboo).

In [ ]:
pivot = by_word_type.pivot(index="word", columns="type", values="mean")
pivot = pivot.reindex(index=WORDS, columns=TYPES)
pivot["Row Mean"] = pivot.mean(axis=1)
col_means = pivot.mean(axis=0)
col_means.name = "Col Mean"
summary = pd.concat([pivot, col_means.to_frame().T])
summary.round(2)


## 4. Visualizations
### 4-1. Mean Tabooness by Word & Sentence Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(WORDS))
n_types = len(TYPES)
w = 0.18
offsets = np.linspace(-(n_types - 1) / 2 * w, (n_types - 1) / 2 * w, n_types)

for i, stype in enumerate(TYPES):
    subset = by_word_type[by_word_type["type"] == stype].set_index("word")
    means = [subset.loc[word, "mean"] if word in subset.index else 0 for word in WORDS]
    sems  = [subset.loc[word, "sem"]  if word in subset.index else 0 for word in WORDS]
    ax.bar(x + offsets[i], means, w, color=TYPE_COLORS[stype], label=stype,
           zorder=3, edgecolor="white", linewidth=0.5)
    ax.errorbar(x + offsets[i], means, yerr=sems,
                fmt="none", color="black", capsize=3, linewidth=1, zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(WORDS, fontsize=11)
ax.set_ylim(1, 7.4)
ax.set_ylabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax.set_title("Mean Tabooness by Word & Sentence Type", fontsize=13, fontweight="bold")
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Sentence Type", fontsize=9, title_fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()


### 4-2. Overall Mean by Sentence Type

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

order = by_type.set_index("type").reindex(TYPES).reset_index()
colors = [TYPE_COLORS[t] for t in order["type"]]
bars = ax.barh(order["type"], order["mean"],
               color=colors, edgecolor="white", height=0.55, zorder=3)
ax.errorbar(order["mean"], order["type"], xerr=order["sem"],
            fmt="none", color="black", capsize=4, linewidth=1.2, zorder=4)

for bar, mean in zip(bars, order["mean"]):
    ax.text(bar.get_width() + 0.20, bar.get_y() + bar.get_height() / 2,
            f"{mean:.2f}", va="center", fontsize=10, fontweight="bold")

ax.set_xlim(1, 8)
ax.set_xlabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax.xaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Overall Mean by Sentence Type", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 4-3. Tabooness Heatmap (Word × Sentence Type)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

pivot_h = by_word_type.pivot(index="word", columns="type", values="mean")
pivot_h = pivot_h.reindex(index=WORDS, columns=TYPES)
data = pivot_h.values

im = ax.imshow(data, cmap="RdYlGn_r", vmin=1, vmax=7, aspect="auto")
ax.set_xticks(range(len(TYPES)))
ax.set_xticklabels(TYPES, fontsize=10)
ax.set_yticks(range(len(WORDS)))
ax.set_yticklabels(WORDS, fontsize=11, fontweight="bold")

for i in range(len(WORDS)):
    for j in range(len(TYPES)):
        ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="white" if data[i, j] > 4.5 or data[i, j] < 2.5 else "black")

plt.colorbar(im, ax=ax, shrink=0.85, label="Mean Rating (1–7)")
ax.set_title("Tabooness Heatmap (Word × Sentence Type)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 4-4. Radar Chart: Sentence Type Profiles Across Words

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

N = len(WORDS)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(WORDS, fontsize=11, fontweight="bold")
ax.set_ylim(1, 7)
ax.set_yticks([2, 3, 4, 5, 6, 7])
ax.set_yticklabels(["2", "3", "4", "5", "6", "7"], fontsize=7, color="grey")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.xaxis.grid(True, linestyle="-", alpha=0.3)
ax.spines["polar"].set_visible(False)

for stype in TYPES:
    subset = by_word_type[by_word_type["type"] == stype].set_index("word")
    vals = [subset.loc[w, "mean"] if w in subset.index else 1 for w in WORDS]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, color=TYPE_COLORS[stype], label=stype)
    ax.fill(angles, vals, alpha=0.08, color=TYPE_COLORS[stype])

ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15),
          title="Sentence Type", fontsize=9, title_fontsize=9)
ax.set_title("Radar: Sentence Type Profiles Across Words",
             fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


### 4-5. Individual Ratings & Word-Level Mean
Each dot is one rating from one respondent. The horizontal line shows the word-level mean.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

rng = np.random.default_rng(42)
for xi, word in enumerate(WORDS):
    subset = long[long["word"] == word]["rating"]
    jitter = rng.uniform(-0.25, 0.25, size=len(subset))
    ax.scatter(np.full(len(subset), xi) + jitter, subset,
               alpha=0.25, s=20, color=WORD_COLORS[word], zorder=2)
    mean_val = subset.mean()
    ax.hlines(mean_val, xi - 0.35, xi + 0.35,
              colors=WORD_COLORS[word], linewidth=3, zorder=3)
    ax.text(xi + 0.38, mean_val, f"{mean_val:.2f}",
            va="center", fontsize=9, fontweight="bold", color=WORD_COLORS[word])

ax.set_xticks(range(len(WORDS)))
ax.set_xticklabels(WORDS, fontsize=11)
ax.set_ylim(1, 7.4)
ax.set_ylabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax.set_title("Individual Ratings & Word-Level Mean", fontsize=13, fontweight="bold")
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

handles = [mpatches.Patch(color=WORD_COLORS[w], label=w) for w in WORDS]
ax.legend(handles=handles, fontsize=9, title="Word", title_fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()


## 5. Save Combined Figure
Render all five plots in a single figure and save as PNG.

In [ ]:
fig = plt.figure(figsize=(18, 16))
fig.suptitle(
    f"Perceived Tabooness of English Swear Words\n"
    f"Survey data · n = {len(raw)} respondents · 1–7 scale",
    fontsize=16, fontweight="bold", y=0.98,
)

gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, :2])
ax4 = fig.add_subplot(gs[1, 2], polar=True)
ax5 = fig.add_subplot(gs[2, :])

# — grouped bar —
x = np.arange(len(WORDS))
offsets = np.linspace(-(len(TYPES)-1)/2*0.18, (len(TYPES)-1)/2*0.18, len(TYPES))
for i, stype in enumerate(TYPES):
    sub = by_word_type[by_word_type["type"]==stype].set_index("word")
    means = [sub.loc[w,"mean"] if w in sub.index else 0 for w in WORDS]
    sems  = [sub.loc[w,"sem"]  if w in sub.index else 0 for w in WORDS]
    ax1.bar(x+offsets[i], means, 0.18, color=TYPE_COLORS[stype], label=stype,
            zorder=3, edgecolor="white", linewidth=0.5)
    ax1.errorbar(x+offsets[i], means, yerr=sems,
                 fmt="none", color="black", capsize=3, linewidth=1, zorder=4)
ax1.set_xticks(x); ax1.set_xticklabels(WORDS, fontsize=11)
ax1.set_ylim(1,7.4); ax1.set_ylabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax1.set_title("Mean Tabooness by Word & Sentence Type", fontsize=13, fontweight="bold")
ax1.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0); ax1.set_axisbelow(True)
ax1.spines[["top","right"]].set_visible(False)
ax1.legend(title="Sentence Type", fontsize=9, title_fontsize=9, loc="upper right")

# — horizontal bar —
order = by_type.set_index("type").reindex(TYPES).reset_index()
bars = ax2.barh(order["type"], order["mean"],
                color=[TYPE_COLORS[t] for t in order["type"]],
                edgecolor="white", height=0.55, zorder=3)
ax2.errorbar(order["mean"], order["type"], xerr=order["sem"],
             fmt="none", color="black", capsize=4, linewidth=1.2, zorder=4)
for bar, mean in zip(bars, order["mean"]):
    ax2.text(bar.get_width()+0.20, bar.get_y()+bar.get_height()/2,
             f"{mean:.2f}", va="center", fontsize=10, fontweight="bold")
ax2.set_xlim(1,8); ax2.set_xlabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax2.xaxis.grid(True, linestyle="--", alpha=0.5, zorder=0); ax2.set_axisbelow(True)
ax2.spines[["top","right"]].set_visible(False)
ax2.set_title("Overall Mean by Sentence Type", fontsize=13, fontweight="bold")

# — heatmap —
pivot_h = by_word_type.pivot(index="word",columns="type",values="mean").reindex(index=WORDS,columns=TYPES)
im = ax3.imshow(pivot_h.values, cmap="RdYlGn_r", vmin=1, vmax=7, aspect="auto")
ax3.set_xticks(range(len(TYPES))); ax3.set_xticklabels(TYPES, fontsize=10)
ax3.set_yticks(range(len(WORDS))); ax3.set_yticklabels(WORDS, fontsize=11, fontweight="bold")
for i in range(len(WORDS)):
    for j in range(len(TYPES)):
        v = pivot_h.values[i,j]
        ax3.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=11,fontweight="bold",
                 color="white" if v>4.5 or v<2.5 else "black")
plt.colorbar(im, ax=ax3, shrink=0.85, label="Mean Rating (1–7)")
ax3.set_title("Tabooness Heatmap (Word × Sentence Type)", fontsize=13, fontweight="bold")

# — radar —
angles = [n/float(len(WORDS))*2*np.pi for n in range(len(WORDS))]
angles += angles[:1]
ax4.set_theta_offset(np.pi/2); ax4.set_theta_direction(-1)
ax4.set_xticks(angles[:-1]); ax4.set_xticklabels(WORDS, fontsize=11, fontweight="bold")
ax4.set_ylim(1,7); ax4.set_yticks([2,3,4,5,6,7])
ax4.set_yticklabels(["2","3","4","5","6","7"], fontsize=7, color="grey")
ax4.yaxis.grid(True,linestyle="--",alpha=0.5); ax4.xaxis.grid(True,linestyle="-",alpha=0.3)
ax4.spines["polar"].set_visible(False)
for stype in TYPES:
    sub = by_word_type[by_word_type["type"]==stype].set_index("word")
    vals = [sub.loc[w,"mean"] if w in sub.index else 1 for w in WORDS] 
    vals += vals[:1]
    ax4.plot(angles, vals, linewidth=2, color=TYPE_COLORS[stype], label=stype)
    ax4.fill(angles, vals, alpha=0.08, color=TYPE_COLORS[stype])
ax4.legend(loc="upper right", bbox_to_anchor=(1.35,1.15),
           title="Sentence Type", fontsize=9, title_fontsize=9)
ax4.set_title("Radar: Sentence Type Profiles Across Words", fontsize=13, fontweight="bold", pad=20)

# — strip plot —
rng = np.random.default_rng(42)
for xi, word in enumerate(WORDS):
    sub = long[long["word"]==word]["rating"]
    jitter = rng.uniform(-0.25,0.25,size=len(sub))
    ax5.scatter(np.full(len(sub),xi)+jitter, sub, alpha=0.25, s=20,
                color=WORD_COLORS[word], zorder=2)
    mv = sub.mean()
    ax5.hlines(mv, xi-0.35, xi+0.35, colors=WORD_COLORS[word], linewidth=3, zorder=3)
    ax5.text(xi+0.38, mv, f"{mv:.2f}", va="center", fontsize=9,
             fontweight="bold", color=WORD_COLORS[word])
ax5.set_xticks(range(len(WORDS))); ax5.set_xticklabels(WORDS, fontsize=11)
ax5.set_ylim(1,7.4); ax5.set_ylabel("Mean Tabooness Rating (1–7)", fontsize=10)
ax5.set_title("Individual Ratings & Word-Level Mean", fontsize=13, fontweight="bold")
ax5.yaxis.grid(True,linestyle="--",alpha=0.5,zorder=0); ax5.set_axisbelow(True)
ax5.spines[["top","right"]].set_visible(False)
ax5.legend(handles=[mpatches.Patch(color=WORD_COLORS[w],label=w) for w in WORDS],
           fontsize=9, title="Word", title_fontsize=9, loc="upper right")

fig.text(0.5, 0.01,
         "Invective = directed insult · Emotive = speaker's feeling · "
         "Informative = neutral / literal context\n"
         "Error bars show ± 1 standard error of the mean.",
         ha="center", fontsize=9, color="gray")

plt.savefig("tabooness_analysis.png", dpi=150, bbox_inches="tight")
print("Figure saved → tabooness_analysis.png")
plt.show()
